# Barriers: is an axis-aligned box the wrong primitive?

Barriers are the one class that will not train. A roadside row receding from the
camera is a long, thin, **diagonal** shape, and an axis-aligned box around it is
mostly empty road — measured on a synthetic row, the oriented box is about **55%**
of the axis-aligned box's area. Nearly half the label is background, which caps
IoU even for a correct prediction.

So: same frames, same filters, same split, same model size, **only the label
geometry differs**.

| run | labels | model |
| --- | --- | --- |
| `B3aabb` | `class cx cy w h` | `yolo11n` |
| `B3obb` | `class x1 y1 x2 y2 x3 y3 x4 y4` | `yolo11n-obb` |

The oriented labels cost nothing extra: the converter already projects 8 cuboid
corners per barrier, so `cv2.minAreaRect` over a structure's corners gives the
tight rotated rectangle directly. No new dataset, unlike segmentation.

Both runs are single-class, so neither number is comparable to the 3-class
`barrier` AP from Task 2 — **only to each other**. That is the point of the
control.

In [ ]:
# ---- edit this ----
NUSCENES = r"C:\path\to\nuscenes-mini"   # same path as the Task 1 notebook
EPOCHS = 60
WORKERS = 6          # drop to 2 if WinError 1455 comes back
DEVICE = ""          # "" = auto, or "cpu", or "0"
# -------------------

In [ ]:
import os, sys
from pathlib import Path

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(REPO)
sys.path.insert(0, str(REPO))

from IPython.display import Image, display

from common.env import find_nuscenes_root
from common.notebook import sh

NUSCENES = find_nuscenes_root(NUSCENES)
DEVICE_ARGS = ["--device", DEVICE] if DEVICE else []
RUNS = {"B3aabb": "configs/barrier_aabb.yaml", "B3obb": "configs/barrier_obb.yaml"}
print("nuScenes:", NUSCENES)

In [ ]:
# Build both datasets off the same pass. Reports how many annotated segments
# collapsed into how many structures - the ratio is the duplicate-label problem
# in one number.
sh("-m", "data.barrier_obb", "--nuscenes-root", NUSCENES)

In [ ]:
# Look at the labels before training on them. Grey = axis-aligned, green = oriented.
import cv2
import numpy as np

from common.env import writable_root

aabb_root = writable_root() / "datasets/barrier_aabb"
obb_root = writable_root() / "datasets/barrier_obb"
dest = REPO / "results/barrier_labels"
dest.mkdir(parents=True, exist_ok=True)

shown = 0
for label in sorted((obb_root / "labels" / "val").glob("*.txt")):
    obb_lines = [l for l in label.read_text().splitlines() if l.strip()]
    if not obb_lines:
        continue
    img_file = next((obb_root / "images" / "val").glob(label.stem + ".*"), None)
    if img_file is None:
        continue
    img = cv2.imread(str(img_file))
    h, w = img.shape[:2]
    for line in (aabb_root / "labels" / "val" / label.name).read_text().splitlines():
        if not line.strip():
            continue
        _, cx, cy, bw, bh = (float(v) for v in line.split())
        p1 = (int((cx - bw / 2) * w), int((cy - bh / 2) * h))
        p2 = (int((cx + bw / 2) * w), int((cy + bh / 2) * h))
        cv2.rectangle(img, p1, p2, (170, 170, 170), 2)
    for line in obb_lines:
        pts = np.array([float(v) for v in line.split()[1:]]).reshape(4, 2)
        pts[:, 0] *= w
        pts[:, 1] *= h
        cv2.polylines(img, [pts.astype(np.int32)], True, (0, 220, 0), 2)
    out = dest / f"{label.stem}.jpg"
    cv2.imwrite(str(out), img)
    display(Image(str(out), width=950))
    shown += 1
    if shown == 3:
        break
print(f"{shown} frames drawn - if the green boxes hug the row and the grey ones do not,")
print("that gap is the whole hypothesis")

In [ ]:
# Train both. The control first, so a crash does not leave only the new thing.
for run in RUNS:
    print()
    print(f"=== {run} ===", flush=True)
    sh(
        "-m", "detection.train", "--run", run,
        "--set", f"epochs={EPOCHS}",
        "--set", f"workers={WORKERS}",
        "--set", "exist_ok=True",
        *DEVICE_ARGS,
    )

In [ ]:
# Score both with ultralytics' own validator, which knows polygon IoU for OBB.
from ultralytics import YOLO

scores = {}
for run, data in RUNS.items():
    weights = REPO / f"runs/{run}/weights/best.pt"
    if not weights.exists():
        print(f"{run}: no weights")
        continue
    res = YOLO(str(weights)).val(data=data, imgsz=640, device=DEVICE or None, verbose=False)
    metric = getattr(res, "box", None) or getattr(res, "obb", None)
    scores[run] = {"mAP50": float(metric.map50), "mAP50-95": float(metric.map)}

print(f"{'run':<10}{'mAP50':>9}{'mAP50-95':>11}")
for run, s in scores.items():
    print(f"{run:<10}{s['mAP50']:>9.3f}{s['mAP50-95']:>11.3f}")

if len(scores) == 2:
    a, o = scores["B3aabb"]["mAP50"], scores["B3obb"]["mAP50"]
    print()
    if o > a + 0.05:
        print(f"oriented boxes win by {o - a:.3f} mAP50 - the primitive was the problem")
    elif a > o + 0.05:
        print("axis-aligned wins; the box shape is not what is holding barriers back")
    else:
        print("too close to call - the limit is elsewhere (data volume, or two scenes)")

In [ ]:
# What each one predicts on held-out frames.
for run in RUNS:
    weights = REPO / f"runs/{run}/weights/best.pt"
    if not weights.exists():
        continue
    model = YOLO(str(weights))
    images = sorted((obb_root / "images" / "val").glob("*.jpg"))
    picks = [p for p in images if (obb_root / "labels" / "val" / f"{p.stem}.txt").read_text().strip()][:2]
    print(f"\n=== {run} ===")
    for path in picks:
        res = model.predict(str(path), imgsz=640, conf=0.25, device=DEVICE or None, verbose=False)[0]
        out = REPO / f"results/barrier_pred_{run}_{path.stem}.jpg"
        cv2.imwrite(str(out), res.plot())
        display(Image(str(out), width=950))

## Reading the result

- **OBB clearly ahead** → the primitive was the problem. Worth carrying into the main
  pipeline: barriers as a separate oriented head, and `D0` reading the oriented box's
  short side (the barrier's height in pixels, independent of how far the row recedes)
  instead of a fitted `H_eff` absorbing that inflation.
- **Both poor** → the box shape was never the limit. Barriers live in two scenes of
  nuScenes-mini and that is simply not enough to generalise, which is then the finding
  to report, with this experiment as the evidence that it was tested properly.
- **Control ahead** → the oriented labels are noisier than they look; check the drawn
  labels above before believing anything else.

---
## The last diagnostic: can it learn barriers at all?

Merged labels, oriented boxes and a single-class setup each changed nothing, so the
box shape was not the limit. One question remains, and it has a different answer
than "the model is wrong":

> can this detector learn barriers **at all**, or only not generalise to a scene it
> has never seen?

A frame-level split answers it: every 5th keyframe of every scene goes to val, so
training and validation both contain barriers from the same scenes.

**This leaks.** Keyframes are 0.5 s apart and show the same barriers from almost the
same angle, so the number is an upper bound, not a generalisation result, and must
never be reported as one. It is a capability check:

- **high here, zero on the scene split** → the detector works; nuScenes-mini's two
  barrier scenes are the limit. That is the finding, and this is the evidence.
- **low here too** → something is still wrong with the barrier labels themselves,
  and the drawn labels above are where to look.

In [ ]:
sh("-m", "data.barrier_obb", "--nuscenes-root", NUSCENES,
   "--split-mode", "frame", "--tag", "frame")

In [ ]:
FRAME_RUNS = {
    "B4aabb": "configs/barrier_frame_aabb.yaml",
    "B4obb": "configs/barrier_frame_obb.yaml",
}
for run in FRAME_RUNS:
    print()
    print(f"=== {run} ===", flush=True)
    sh(
        "-m", "detection.train", "--run", run,
        "--set", f"epochs={EPOCHS}",
        "--set", f"workers={WORKERS}",
        "--set", "exist_ok=True",
        *DEVICE_ARGS,
    )

In [ ]:
for run, data in FRAME_RUNS.items():
    weights = REPO / f"runs/{run}/weights/best.pt"
    if not weights.exists():
        print(f"{run}: no weights")
        continue
    res = YOLO(str(weights)).val(data=data, imgsz=640, device=DEVICE or None, verbose=False)
    metric = getattr(res, "box", None) or getattr(res, "obb", None)
    scores[run] = {"mAP50": float(metric.map50), "mAP50-95": float(metric.map)}

print(f"{'run':<10}{'split':<8}{'mAP50':>9}{'mAP50-95':>11}")
for run, s in scores.items():
    kind = "frame" if run.startswith("B4") else "scene"
    print(f"{run:<10}{kind:<8}{s['mAP50']:>9.3f}{s['mAP50-95']:>11.3f}")

frame_best = max((s["mAP50"] for r, s in scores.items() if r.startswith("B4")), default=0)
scene_best = max((s["mAP50"] for r, s in scores.items() if r.startswith("B3")), default=0)
print()
if frame_best > 0.4 and scene_best < 0.1:
    print("The detector learns barriers fine. What it cannot do is transfer to a scene")
    print("it has never seen - and nuScenes-mini has barriers in only two scenes.")
    print("That is a dataset limit, and these two numbers together are the evidence.")
elif frame_best <= 0.4:
    print("It cannot learn barriers even with leakage, so the labels are still wrong.")
    print("Go back to the drawn labels above before changing anything else.")
else:
    print("Both splits score; barriers are fine after all.")